In [1]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


In [2]:
!pip install -q transformers

In [3]:
from google.colab import files

uploaded = files.upload()

Saving clean_comments.csv to clean_comments.csv


In [4]:
import pandas as pd

clean_df = pd.read_csv("clean_comments.csv")

print(clean_df.shape)

clean_df[
    ["comment_id", "entry_id", "comment_clean"]
].head()

(110336, 8)


,comment_id,entry_id,comment_clean
0,e/624ca9226b6526ebdb69f9b46df482c7/c/32c6bf5bc...,e/624ca9226b6526ebdb69f9b46df482c7,Reel Review video: Catherine Shoard defends Kn...
1,e/967b4db48fa74021b24ccbc93c55a61c/c/57905e983...,e/967b4db48fa74021b24ccbc93c55a61c,Article by at 2010-08-06 09:42:14 Categorized ...
2,e/79e585effdf640e988539e2dd68c2c6d/c/853946596...,e/79e585effdf640e988539e2dd68c2c6d,Happy Friday! @Skyblue101 and @TropicsZ4: Matc...
3,e/01e9a149fff85ff948972896145d3d65/c/e87986e73...,e/01e9a149fff85ff948972896145d3d65,Yesterday the enviously green city of Portland...
4,e/01e9a149fff85ff948972896145d3d65/c/c70184cfd...,e/01e9a149fff85ff948972896145d3d65,Yesterday the enviously green city of Portland...


In [5]:
import torch
import pandas as pd

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

model_name = "j-hartmann/emotion-english-distilroberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)
model.eval()

print("Using device:", device)
print(model.config.id2label)

config.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  329MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Using device: cuda
{0: 'anger', 1: 'disgust', 2: 'fear', 3: 'joy', 4: 'neutral', 5: 'sadness', 6: 'surprise'}


In [6]:
from tqdm.auto import tqdm

texts = (
    clean_df["comment_clean"]
    .fillna("")
    .astype(str)
    .tolist()
)

batch_size = 64

all_probabilities = []

for i in tqdm(
    range(0, len(texts), batch_size),
    desc="Emotion detection"
):

    batch_texts = texts[i:i + batch_size]

    inputs = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(
        outputs.logits,
        dim=1
    )

    all_probabilities.extend(
        probs.cpu().numpy()
    )

Emotion detection:   0%|          | 0/1724 [00:00<?, ?it/s]

In [7]:
emotion_columns = [
    "anger",
    "disgust",
    "fear",
    "joy",
    "neutral",
    "sadness",
    "surprise"
]

emotion_df = pd.DataFrame(
    all_probabilities,
    columns=emotion_columns
)

emotion_df.head()

,anger,disgust,fear,joy,neutral,sadness,surprise
0,0.010169,0.008245,0.003183,0.428729,0.490051,0.004336,0.055287
1,0.004119,0.001753,0.002665,0.005910,0.852355,0.019294,0.113904
2,0.003618,0.000299,0.001818,0.844965,0.027604,0.005352,0.116344
3,0.013171,0.016564,0.008067,0.069312,0.752039,0.004490,0.136357
4,0.013171,0.016564,0.008067,0.069312,0.752039,0.004490,0.136357


In [8]:
emotion_results = pd.concat(
    [
        clean_df[
            [
                "comment_id",
                "entry_id",
                "user",
                "timestamp",
                "comment_clean"
            ]
        ].reset_index(drop=True),

        emotion_df
    ],
    axis=1
)

In [10]:
emotion_results["Top_Emotion"] = (
    emotion_results[
        emotion_columns
    ].idxmax(axis=1)
)

emotion_results["Top_Emotion_Score"] = (
    emotion_results[
        emotion_columns
    ].max(axis=1)
)

emotion_results[
    [
        "comment_clean",
        "Top_Emotion",
        "Top_Emotion_Score"
    ]
].head(10)

,comment_clean,Top_Emotion,Top_Emotion_Score
0,Reel Review video: Catherine Shoard defends Kn...,neutral,0.490051
1,Article by at 2010-08-06 09:42:14 Categorized ...,neutral,0.852355
2,Happy Friday! @Skyblue101 and @TropicsZ4: Matc...,joy,0.844965
3,Yesterday the enviously green city of Portland...,neutral,0.752039
4,Yesterday the enviously green city of Portland...,neutral,0.752039
5,“The founders viewed the criminal sanction as ...,neutral,0.351657
6,&quot;Mr. Ryan has become the Republican Party...,joy,0.794975
7,She does need a break - and Veronica will be w...,neutral,0.615171
8,&quot;Lake Nakuru Park and the Masai Mara See ...,joy,0.870483
9,Article by at 2010-08-06 10:00:43 Categorized ...,neutral,0.861669


In [13]:
emotion_results["Top_Emotion"].unique()

array(['neutral', 'joy', 'disgust', 'fear', 'sadness', 'anger',
       'surprise'], dtype=object)

In [14]:
emotion_results["Top_Emotion"].value_counts()

,count
Top_Emotion,
neutral,71585
joy,12575
surprise,8848
sadness,6571
anger,4033
fear,3826
disgust,2898


In [11]:
emotion_results.to_csv(
    "emotion_results.csv",
    index=False
)

print("Saved shape:", emotion_results.shape)

Saved shape: (110336, 14)


In [12]:
from google.colab import files

files.download("emotion_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>